# PyTorch Image Classification

## Setup
## Imports

In [ ]:
# Set up Kaggle API credentials for downloading datasets
import os  # For file and directory operations
import shutil  # For file copying

kaggle_json_path = './kaggle.json'  # Path to your Kaggle API key
kaggle_dir = os.path.expanduser('~/.kaggle')  # Default Kaggle directory
os.makedirs(kaggle_dir, exist_ok=True)  # Create directory if it doesn't exist
shutil.copy(kaggle_json_path, os.path.join(kaggle_dir, 'kaggle.json'))  # Copy API key
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)  # Set permissions
print('Kaggle API key set up successfully.')

Kaggle API key set up successfully.


In [ ]:
# Install and import opendatasets, then download the animal faces dataset from Kaggle
!pip install opendatasets --quiet  # Install opendatasets if not already installed
import opendatasets as od  # Import opendatasets for easy dataset download
od.download("https://www.kaggle.com/datasets/andrewmvd/animal-faces")  # Download dataset

Skipping, found downloaded files in ".\animal-faces" (use force=True to force download)


In [ ]:
# Import required libraries and set device
import torch  # PyTorch main package
print(f"PyTorch version: {torch.__version__}")
import torch.nn as nn  # Neural network layers
from torch.optim import Adam  # Optimizer
from torch.utils.data import Dataset, DataLoader  # Dataset and DataLoader utilities
from sklearn.preprocessing import LabelEncoder  # For encoding string labels
import numpy as np  # Numerical operations
from torchvision import transforms  # Image transformations
import matplotlib.pyplot as plt  # Plotting

torch.manual_seed(17)  # For reproducibility
import pandas as pd  # Data manipulation
from PIL import Image  # Image processing

# Set device to GPU if available, else CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PyTorch version: 2.9.1+rocm7.2.1
Using device: cuda


## Image data classification with PyTorch

In [ ]:
# Collect image file paths and labels from the dataset directory
image_path = []
labels = []

# Loop through each split (train/val) and each class (cat/dog/wild)
for i in os.listdir('animal-faces/afhq'):
    for label in os.listdir(f'animal-faces/afhq/{i}'):
        for img in os.listdir(f'animal-faces/afhq/{i}/{label}'):
            image_path.append(f'animal-faces/afhq/{i}/{label}/{img}')  # Store image path
            labels.append(label)  # Store label
            
# Create a DataFrame with image paths and labels
data_df = pd.DataFrame(zip(image_path, labels), columns=['image_path', 'label'])
print(data_df["label"].unique())  # Show unique labels
data_df.head()  # Display first few rows

['cat' 'dog' 'wild']


,image_path,label
0,animal-faces/afhq/train/cat/flickr_cat_000002.jpg,cat
1,animal-faces/afhq/train/cat/flickr_cat_000003.jpg,cat
2,animal-faces/afhq/train/cat/flickr_cat_000004.jpg,cat
3,animal-faces/afhq/train/cat/flickr_cat_000005.jpg,cat
4,animal-faces/afhq/train/cat/flickr_cat_000006.jpg,cat


In [ ]:
# Split the data into train, validation, and test sets
train = data_df.sample(frac=0.7, random_state=42)  # 70% for training
test = data_df.drop(train.index)  # Remaining 30%

val = test.sample(frac=0.5, random_state=42)  # Half of remaining for validation
test = test.drop(val.index)  # The rest for testing

print(f"Train size: {len(train)}, Val size: {len(val)}, Test size: {len(test)}")

Train size: 11291, Val size: 2420, Test size: 2419


In [ ]:
# Encode string labels to integers
label_encoder = LabelEncoder()
label_encoder.fit(data_df["label"])

# Define image transformations for preprocessing
# - Resize images to 128x128
# - Convert images to PyTorch tensors
# - Ensure tensor dtype is float
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float),
])

In [ ]:
# Custom PyTorch Dataset for loading images and labels
def __init__(self, df, transform=None):
    self.df = df  # DataFrame with image paths and labels
    self.transform = transform  # Image transformations
    
def __len__(self):
    return len(self.df)  # Number of samples

def __getitem__(self, idx):
    img_path = self.df.iloc[idx]['image_path']  # Get image path
    label = self.df.iloc[idx]['label']  # Get label
    
    image = Image.open(img_path).convert('RGB')  # Open and convert image to RGB
    
    if self.transform:
        image = self.transform(image)  # Apply transformations
    
    label = label_encoder.transform([label])[0]  # Encode label as integer
    
    return image, label

class CustomImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df  # DataFrame with image paths and labels
        self.transform = transform  # Image transformations
        
    def __len__(self):
        return len(self.df)  # Number of samples
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']  # Get image path
        label = self.df.iloc[idx]['label']  # Get label
        
        image = Image.open(img_path).convert('RGB')  # Open and convert image to RGB
        
        if self.transform:
            image = self.transform(image)  # Apply transformations
        
        label = label_encoder.transform([label])[0]  # Encode label as integer
        
        return image, label